# Import necessary libraries for AWS and SageMaker

In [1]:
import boto3
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


# Set up AWS sessions and get the default SageMaker S3 bucket.

In [8]:
session = boto3.session.Session()
region = session.region_name
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()
bucket

'sagemaker-us-east-1-730335647345'

Create an S3 client to interact with Amazon S3 in the current region.

In [6]:
s3 = boto3.Session().client(service_name="s3", region_name=region)

# Define S3 path and upload the local CSV file to the specified S3 bucket and folder.

In [48]:
s3_private_path_csv = f's3://{bucket}/homeword_dataset_csv'
s3_key = '2-1-dataset.csv'
with open('data/dataset.csv') as f:
    s3.put_object(Bucket=bucket, Key=f'homeword_dataset_csv/{s3_key}', Body=f.read())

# Set up Athena connection and create database if needed.

In [19]:
from pyathena import connect

In [20]:
database_name = "homework21"
s3_staging_dir = "s3://{0}/homework21/athena/staging".format(bucket)
conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

In [21]:
statement = "CREATE DATABASE IF NOT EXISTS {}".format(database_name)

In [22]:
import pandas as pd

pd.read_sql(statement, conn)

/tmp/ipykernel_2647/3803073958.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


In [35]:
pd.read_csv('data/dataset.csv').dtypes

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
dtype: object

In [69]:
table_name_csv = 'homeword_dataset_csv'

statement = """CREATE EXTERNAL TABLE IF NOT EXISTS {}.{}(
index               int,
track_id             string,
artists              string,
album_name           string,
track_name           string,
popularity            int,
duration_ms           int,
explicit               boolean,
danceability        float,
energy              float,
key                   int,
loudness            float,
mode                  int,
speechiness         float,
acousticness        float,
instrumentalness    float,
liveness            float,
valence             float,
tempo               float,
time_signature        int,
track_genre          string
) ROW FORMAT DELIMITED FIELDS TERMINATED BY ',' LINES TERMINATED BY '\\n' LOCATION '{}'
TBLPROPERTIES ('compressionType'='gzip', 'skip.header.line.count'='1')""".format(
    database_name, table_name_csv, s3_private_path_csv
)

pd.read_sql(statement, conn)

/tmp/ipykernel_2647/832900183.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(statement, conn)


""


# Create an external table in Athena from the CSV file in S3

In [71]:
statement = """SELECT * FROM {}.{} LIMIT 100""".format(
    database_name, table_name_csv
)

print(statement)

df = pd.read_sql(statement, conn)
df.head(5)

SELECT * FROM homework21.homeword_dataset_csv LIMIT 100


/tmp/ipykernel_2647/1678997614.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


,index,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73.0,230666.0,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55.0,149610.0,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57.0,210826.0,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71.0,201933.0,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82.0,198853.0,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [72]:
statement = """
SELECT
track_name,
energy
FROM {}.{}
WHERE energy >= 0.5
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)

/tmp/ipykernel_2647/2008175252.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


                             track_name  energy
0                                Hunger   0.632
1                       Hold On - Remix   0.780
2      Falling in Love at a Coffee Shop   0.561
3                               Vol. 4"   0.717
4                               Vol. 3"   0.678
...                                 ...     ...
81862        Stay (You Are Good) - Live   0.762
81863       At The Cross (Love Ran Red)   0.531
81864             Your Love Never Fails   0.860
81865       How Can I Keep From Singing   0.687
81866                           Friends   0.506

[81867 rows x 2 columns]


# List artist, track_name, and popularity for songs that have a popularity greater than or equal to 99

In [73]:
statement = """
SELECT
  artists,
  track_name,
  popularity
FROM {}.{}
WHERE popularity >= 99
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)


/tmp/ipykernel_2647/266582748.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


                artists                 track_name  popularity
0  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)         100
1     Charlie Brown Jr.               Prazo Longo"         333
2          Smyang Piano                    Vol. 4"      134340
3  Sam Smith;Kim Petras  Unholy (feat. Kim Petras)         100


# List artists with an average popularity of 92

In [74]:
statement = """
SELECT
  artists,
  AVG(popularity) AS avg_popularity
FROM {}.{}
GROUP BY artists
HAVING AVG(popularity) = 92
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)


/tmp/ipykernel_2647/3347748660.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


             artists  avg_popularity
0  Rema;Selena Gomez            92.0
1       Harry Styles            92.0


# List the Top 10 genres with the highest average energy

In [75]:
statement = """
SELECT
  track_genre,
  AVG(energy) AS avg_energy
FROM {}.{}
GROUP BY track_genre
ORDER BY avg_energy DESC
LIMIT 10
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)


/tmp/ipykernel_2647/1207464896.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


  track_genre  avg_energy
0       0.797   1174026.0
1       0.556    691306.0
2      0.0371    629420.0
3      0.0359    614791.0
4       0.492    542000.0
5        0.45    538160.0
6       0.914    531293.0
7      0.0427    526946.0
8      0.0761    502786.0
9      0.0346    500088.0


# How many tracks is Bad Bunny on?

In [76]:
statement = """
SELECT
  COUNT(*) AS track_count
FROM {}.{}
WHERE artists LIKE '%Bad Bunny%'
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)


/tmp/ipykernel_2647/348264072.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


   track_count
0          416


# Show the top 10 genres in terms of popularity, sorted by their most popular track

In [77]:
statement = """
SELECT
  track_genre,
  MAX(popularity) AS max_popularity
FROM {}.{}
GROUP BY track_genre
ORDER BY max_popularity DESC
LIMIT 10
""".format(database_name, table_name_csv)
df = pd.read_sql(statement, conn)
print(df)


/tmp/ipykernel_2647/1824109531.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(statement, conn)


  track_genre  max_popularity
0           4          134340
1       dance             100
2         pop             100
3       latin              98
4      latino              98
5      reggae              98
6   reggaeton              98
7         edm              98
8        rock              96
9       piano              96
